# exp03: Normalisasi dan Missing Values

Menguji nilai absolut, transformasi log, centering kausal, scaler, missing indicator, native NaN, serta winsorization training-only. Perlakuan dibandingkan satu faktor terhadap kontrolnya.


## Akses Proyek

Menentukan root lokal atau Google Drive. Semua notebook memakai paket batch-02 yang sama; hanya lokasi proyek yang perlu disesuaikan.


In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount('/content/drive')

PROJECT_ROOT = None
if PROJECT_ROOT is None:
    if IN_COLAB:
        PROJECT_ROOT = Path('/content/drive/MyDrive/Penelitian-Adaptive-IDE')
    else:
        PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                             if (p / 'models' / 'batch-02').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Isi PROJECT_ROOT dengan folder proyek yang memuat models/batch-02.')
PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
BATCH_DIR = PROJECT_ROOT / 'models' / 'batch-02'
if not (BATCH_DIR / 'shared' / 'batch02').is_dir():
    raise RuntimeError('Salin folder batch-02 lengkap, termasuk shared dan requirements.txt.')
sys.path.insert(0, str(BATCH_DIR / 'shared'))


## Environment

Menyiapkan dependency inti sebelum memuat modul data. Instalasi hanya dilakukan bila versi berbeda. Dependency neural dan TabPFN bersifat opsional dan tidak mengunduh checkpoint. Jika instalasi mengubah modul yang sudah dimuat, restart runtime sekali lalu jalankan dari awal.


In [ ]:
from batch02.runtime import ensure_environment

INSTALL_DEPENDENCIES = IN_COLAB
INSTALL_OPTIONAL = False
OPTIONAL_GROUPS = ()
environment_status = ensure_environment(
    BATCH_DIR, install=INSTALL_DEPENDENCIES,
    optional=OPTIONAL_GROUPS if INSTALL_OPTIONAL else (),
)
print(environment_status)


## Konfigurasi

Menetapkan sumber data dan output tanpa mengunci jumlah peserta atau distribusi kelas. Seed mengikuti protokol batch-02. Tidak ada kewajiban menjalankan mode smoke sebelum evaluasi penelitian.


In [ ]:
from batch02.contracts import Config

EXPERIMENT = 'exp03'
DATASET_ROOT = None
RESUME_DIR = None
config = Config(PROJECT_ROOT, dataset_root=DATASET_ROOT)
print('Dataset:', config.dataset)
print('Output:', config.batch / 'outputs' / EXPERIMENT)


## Input Release dan Pembagian Data

Memuat snapshot yang sudah disahkan. Training berhenti sebelum fitting jika gate belum lengkap atau hash input berubah. Semua sesi orang yang sama tetap satu grup; identitas hanya mengatur evaluasi.


In [ ]:
from batch02.contracts import load_release

release, frames, feature_allowlist, split_manifest = load_release(config)
display(release['summary'])
print('Release:', release['release_id'])
print('Fitur sensor:', len(feature_allowlist))


## Kandidat dan Rancangan Evaluasi

Menampilkan perlakuan yang akan diuji sebelum training. Pemilihan parameter menggunakan inner validation. Macro-F1 dari pooled outer OOF menjadi metrik utama; accuracy, precision, recall, dan confusion matrix melengkapinya.


In [ ]:
from batch02.runner import describe_experiment
from batch02.registry import experiment_windows

candidate_table = describe_experiment(EXPERIMENT)
display(candidate_table)
print('Window:', experiment_windows(EXPERIMENT))


## Training dan Evaluasi

Cell ini menjalankan fitting dan evaluasi penelitian pada release final. Preprocessing, seleksi, balancing, serta stopping mengikuti fitting partition. Hasil disimpan per fold; RESUME_DIR hanya boleh menunjuk run dengan identitas data/kode/environment yang sama.


In [ ]:
from batch02.runner import run_experiment

run_dir, results = run_experiment(config, EXPERIMENT, resume=RESUME_DIR)
print('Direktori hasil:', run_dir)
display(results)


## Hasil Tersimpan

Ringkasan berikut dibaca dari prediksi yang benar-benar dihasilkan. Cabang skipped tidak diberi skor nol; OOF parsial tidak diperlakukan sebagai evaluasi lengkap. SVM mengekspor decision score, bukan probabilitas.


In [ ]:
from batch02.artifacts import read_json

status = read_json(run_dir / 'status.json')
display(status)
if (run_dir / 'metrics.json').is_file():
    display(read_json(run_dir / 'metrics.json'))
if (run_dir / 'joint_window_metrics.json').is_file():
    display(read_json(run_dir / 'joint_window_metrics.json'))
